# GRS-34806 MGI Project
## 1) Download the data
This template downloads the UCM data (both mono and multi-labels) in your local Colab environment.

Write the notebook in such a way that it fully runs from start to end without further intervention  
(i.e. do not change the directory structure manually in the mean time).

Good luck!

In [1]:
import os
import zipfile

! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
    zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md  UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

Cloning into 'ucmdata'...
remote: Enumerating objects: 8, done.
remote: Total 8 (delta 0), reused 0 (delta 0), pack-reused 8 (from 1)
Receiving objects: 100% (8/8), 316.99 MiB | 14.21 MiB/s, done.
Images	LandUse_Multilabeled.txt


In [2]:
## add Weights & Biases for logging
#%pip install -q wandb
#import wandb
#wandb.login()

In [3]:
## Setting up our images
import os
dir = "/content/ucmdata/Images"
image_dict = {}
class_idx_dict = {}


for idx, class_name in enumerate(os.listdir(dir)):
  class_path = os.path.join(dir, class_name)
  class_idx_dict[class_name] = idx

  if os.path.isdir(class_path):
    image_paths = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.endswith(('.tif'))]
    image_dict[idx] = image_paths

In [4]:

import random

def create_datasets(image_dict, val_fraction, test_fraction):
    train_dataset = []
    val_dataset = []
    test_dataset = []

    for label, paths in image_dict.items():
        random.shuffle(paths)

        # First split: (train+val) - test
        split_idx = int(len(paths) * (1 - test_fraction))
        temp_paths = paths[:split_idx]
        test_paths = paths[split_idx:]

        # Second split: train - val (from temp_paths)
        test_split_idx = int(len(temp_paths) * (1-val_fraction))
        train_paths = temp_paths[:test_split_idx]
        val_paths = temp_paths[test_split_idx:]

        train_dataset.extend([(path, label) for path in train_paths])
        val_dataset.extend([(path, label) for path in val_paths])
        test_dataset.extend([(path, label) for path in test_paths])

    return train_dataset, val_dataset, test_dataset

In [5]:
train_data, val_data, test_data = create_datasets(image_dict, 0.2, 0.2)

In [6]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class UCMDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader

# transformations
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # descriptive stats from ImageNet
])

In [8]:
train_ds = UCMDataset(train_data, transform=transform)
val_ds = UCMDataset(val_data, transform=transform)
test_ds = UCMDataset(test_data, transform=transform)

In [9]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# 1. First Approach: Pretrained ResNet18 and finetune

The following code is an implementation of the medium post here:
https://medium.com/@imabhi1216/fine-tuning-a-pre-trained-resnet-18-model-for-image-classification-on-custom-dataset-with-pytorch-02df12e83c2c

It uses a pretrained ResNet18 and finetunes the head for our specific case

In [10]:
import torch
import torchvision.models as models

# Load the pre-trained ResNet-18 model
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 101MB/s]


In [11]:
# Modify the last layer of the model
num_classes = 21 # replace with the number of classes in your dataset
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

In [12]:
# Define the loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

In [13]:
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            print(inputs.size(0))
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # Print the results for the current epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train(model, train_loader, val_loader, criterion, optimizer, num_epochs=5)

Epoch [1/5], train loss: 2.8373, train acc: 0.1741, val loss: 2.2781, val acc: 0.4167
Epoch [2/5], train loss: 1.8818, train acc: 0.6101, val loss: 1.5583, val acc: 0.6667
Epoch [3/5], train loss: 1.3644, train acc: 0.7604, val loss: 1.1776, val acc: 0.7827
Epoch [4/5], train loss: 1.0639, train acc: 0.8385, val loss: 0.9650, val acc: 0.8333
Epoch [5/5], train loss: 0.8694, train acc: 0.8728, val loss: 0.8158, val acc: 0.8482
